# Deep Learning / PyTorch

FC-сети, Dataset/DataLoader, train/eval loop, оптимизаторы, schedulers, save/load.

## Импорты + seed

In [ ]:
import random
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Dataset и DataLoader

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.features[idx], self.labels[idx]
        return self.features[idx]

# X_train, y_train, X_val, y_val -- numpy arrays
train_ds = TabularDataset(X_train, y_train)
val_ds = TabularDataset(X_val, y_val)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

## FC-модель

In [ ]:
class FCModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)

INPUT_DIM = X_train.shape[1]  # число признаков
HIDDEN_DIM = 256
NUM_CLASSES = len(np.unique(y_train))

model = FCModel(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Оптимизатор, loss, scheduler

In [ ]:
NUM_EPOCHS = 50
LR = 1e-3

criterion = nn.CrossEntropyLoss()  # label_smoothing=0.1
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Scheduler (выбрать один)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

## Train loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1

## Eval loop

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1, np.array(all_preds)

## Основной цикл обучения

In [ ]:
best_f1 = 0
patience = 7
counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_f1, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()  # вряд ли нужно, скорее если вы добираете последние баллы

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d} | train_loss={train_loss:.4f} train_f1={train_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'\nBest val F1: {best_f1:.4f}')
model.load_state_dict(torch.load('best_model.pt', map_location=device))

## Предсказание на тесте

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_preds = []
    for batch_x in loader:
        if isinstance(batch_x, (list, tuple)):
            batch_x = batch_x[0]
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
    return np.array(all_preds)

test_ds = TabularDataset(X_test_np)  # X_test -- numpy array без лейблов
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_preds = predict(model, test_loader, device)

## Save / Load

In [ ]:
# Сохранить
torch.save(model.state_dict(), 'model_weights.pt')

# Загрузить
model = FCModel(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES).to(device)
model.load_state_dict(torch.load('model_weights.pt', map_location=device))
model.eval()